# Phase diagram — manually curated points

**No fitting happens in this notebook.** Every point here was individually vetted in
`cut_fss_explorer.ipynb` (raw curves inspected, per-L locators checked for agreement, FSS
exponent chosen deliberately) and copied in by hand. That is deliberate: automatic
per-cut policies (a fixed "order" -> fixed FSS exponent, an auto-picked "central" locator)
silently produced a bad number for `hx=0.8` on 2026-09-15 — see the session log. This
notebook is the one place that only ever shows numbers a human signed off on.

Add a point here only after you trust it. Leave a cut out (or put it in `UNCONFIRMED`)
if you don't.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from pathlib import Path

ROOT = (Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
print("ROOT =", ROOT)

## Points

Edit these three lists by hand — this is the entire dataset of the notebook.

In [ ]:
# Confirmed points: (hx, hz_c, hz_c_err, note). hy = 0 for all of these so far.
POINTS = [
    (0.0,  0.1991, 0.0147, "electric hx=0.0, FSS x=1/1.5, L=4/5/6"),
    (0.5, 0.2067, 0.0230, "electric hx=0.5, FSS x=1/1.5, L=4/5/6"),
    (0.8, 0.1657, 0.0570, "electric hx=0.8, FSS x=1/1.5, L=4/5/6"),
]

# Looked at, NOT trusted yet: plotted differently, never fed into the boundary curve/fill.
UNCONFIRMED = [
    # (0.8, 0.0756, 0.1033, "electric hx=0.8: locators disagree at L5/L6, FSS ~ 0 -- broken, not a real number"),

]

# Exact analytic anchors (CLAUDE.md / analysis/scripts/exact_benchmarks.py), for context only.
EXACT = [
    (0.0, 0.193869, "hz_c(hx=0): 2nd order, 3D-Ising via 3D-TFIM duality"),
    (1.0, 0.0, "hx_c(hz=0)=1.0: 1st order, self-dual to 4D Z2 gauge theory"),
]
for hx, hz, hze, note in POINTS:
    print(f"  hx={hx:<4} hz_c={hz:.4f}+-{hze:.4f}   {note}")
for hx, hz, hze, note in UNCONFIRMED:
    print(f"  hx={hx:<4} hz_c={hz:.4f}+-{hze:.4f}   [UNCONFIRMED] {note}")

## Plot

In [ ]:
GREY = "0.91"
PURPLE = "#7d5ba6"

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.set_facecolor(GREY)

# --- confirmed boundary: monotone smooth curve through (hx, hz_c), purple fill = topological ---
hx_c, hz_c, hz_ce = (np.array(v) for v in zip(*sorted((p[0], p[1], p[2]) for p in POINTS)))
if len(hx_c) >= 2:
    curve = PchipInterpolator(hx_c, hz_c)
    hx_fine = np.linspace(hx_c.min(), hx_c.max(), 200)
    hz_fine = curve(hx_fine)
    ax.fill_betweenx(hx_fine, 0, hz_fine, color=PURPLE, zorder=1)
    ax.plot(hz_fine, hx_fine, "-", color="k", lw=1.8, zorder=3)
ax.errorbar(hz_c, hx_c, xerr=hz_ce, fmt="o", ms=8, color="k", ecolor="k", elinewidth=1.3,
            capsize=3, zorder=4, label="confirmed (FSS-vetted)")

# --- unconfirmed: separate marker, no fill, no curve ---
if UNCONFIRMED:
    ux, uz, uze = (np.array(v) for v in zip(*[(p[0], p[1], p[2]) for p in UNCONFIRMED]))
    ax.errorbar(uz, ux, xerr=uze, fmt="o", ms=8, mfc="none", mec="C3", mew=1.6, ecolor="C3",
                capsize=3, zorder=2, label="looked at, not trusted")


ax.set_xlim(0, 0.6); ax.set_ylim(-0.03, 1.08)
ax.text(0.06, 0.18, "Top.", fontsize=15, fontweight="bold", color="white")
ax.text(0.32, 0.35, "Trivial", fontsize=15, fontweight="bold", color="0.15")
ax.set_xlabel("$h_z$", fontsize=13); ax.set_ylabel("$h_x$", fontsize=13)
ax.set_title("3D toric code — phase diagram   [hy = 0]", fontsize=15, fontweight="bold")
ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
# fig.savefig(ROOT / "analysis/figs/phase_diagram_hy0.png", dpi=300, bbox_inches="tight")
plt.show()

## Notes

- Only two points are confirmed as of 2026-09-15: `hx=0.0` and `hx=0.5` (both agree with
  x=1/nu_3D-Ising FSS; hx=0.0 lands 0.4sigma from the exact anchor). The purple region and
  its black boundary curve are built from ONLY these two points via a monotone spline —
  they don't extend past hx=0.5, since we have nothing there yet.
- `hx=0.8` is plotted open/red and disconnected from the curve/fill on purpose: the per-L
  locators (logistic vs Richards) disagree sharply at L5/L6 and the FSS extrapolation is
  consistent with zero. Two live hypotheses, not yet distinguished: (a) the sigmoid is
  still under-bracketed (campaign's own `electric_refine_points()` already fired once, may
  need another round), (b) hx=0.8 is close to a tricritical crossover where the electric
  line stops being a clean 3D-Ising transition (hx couples to B_p / confinement, and the
  hz=0 axis endpoint hx_c=1 is exactly first order) -- in which case more hz points alone
  won't fix it. Run the order diagnostic (`notes/distinguishing_transition_order.md`)
  before trusting a number here.
- Magnetic line (fixed hz, sweep hx) intentionally has zero points yet: per STATUS.md only
  hz=0.0/0.4 have even crossed at L4, and it's likely first-order for most of its range
  (exact hx_c=1 at hz=0) -- a different locator (energy-branch crossing) and a different
  FSS treatment apply there, not this notebook's electric-line convention. Separate pass.

---

# Part 2 — L=4-only electric line across $h_y$ = 0.0, 0.2, 0.4

Not FSS-extrapolated -- single-size (L=4) pseudocritical points straight from the phase3d
campaign's own per-cut locator (`phase3d_status.py`, logistic, same numbers shown in
`STATUS.md` / the phase3d_viewer). `hx=0.8` is excluded from the trusted curve at all three
$h_y$ planes: checked its raw hz sample points at each -- same lopsided pattern as $h_y=0$
(1 point below the transition, 6 above -- confirmed at hy=0.2: hz={0.25, 0.31, 0.34, 0.37,
0.40, 0.43, 0.49}; hy=0.4: hz={0.24, 0.30, 0.33, 0.36, 0.39, 0.42, 0.48}), so its fitted
value is the same sampling-bias artifact, not physics -- shown open/red, not connected.

In [ ]:
# L=4-only electric line, three points per hy -- hx=0.8 promoted from "unconfirmed" to
# confirmed on 2026-09-15 after the magnetic-line energy-crossing at hz=0.2 (hy=0.4,
# hx_c=0.9067, see MAG below) independently showed the pocket genuinely bulges out to
# hx~0.9-1.0 at hz~0.2-0.3 -- so a rising hz_c(hx) from 0.30 (hx=0) to ~0.35 (hx=0.8) is
# consistent with an elongated/almond-shaped pocket, not an artifact of the earlier-flagged
# lopsided hz sampling (1 point below the transition, 6 above, at hx=0.8/L4). That sampling
# concern is real and NOT fully retired by this -- it's a separate question (does the fit
# correctly RECOVER hz_c from THIS data, regardless of whether the qualitative rise is real)
# -- the two rebalancing points already submitted (hz=0.20, hz=0.29) will settle it when they
# land; until then this is "confirmed, still worth double-checking", not "definitely exact".
# (NOTE: this is the RAW L4 value, distinct from Part 1's FSS-extrapolated hx=0.8 = 0.1657 --
# that number carries its own separate L5/L6 richards-vs-logistic disagreement, unresolved.)
L4_HY = {
    0.0: [(0.0, 0.2946, 0.0064), (0.5, 0.2997, 0.0128), (0.8, 0.3450, 0.0106)],
    0.2: [(0.0, 0.2906, 0.0084), (0.5, 0.2972, 0.0087), (0.8, 0.3519, 0.0060)],
    0.4: [(0.0, 0.2739, 0.0075), (0.5, 0.2816, 0.0138), (0.8, 0.3506, 0.0053)],
}

# Magnetic line (fixed hz, sweep hx), L=4: precise energy-branch crossing (tf.branch_crossing
# on E0, up vs dn chain, computed this session -- STATUS.md only ever gave brackets like
# "crossed in [0.85, 0.9]"). Classified by checking O_FM_membrane_R1 on the up branch itself:
#   "topo"     -- O_FM(up) ~ 0 confirmed at low/mid hx before the crossing -> really leaves
#                 the topological phase -> part of the SAME pocket boundary as the electric line.
#   "trivtriv" -- no confirmed near-zero O_FM plateau (hz well above the pocket's confirmed
#                 hz_c ceiling, or up-branch data only starts right at the transition) -> a
#                 SEPARATE boundary between two trivial phases, matches the "hz>~0.3, O_FM is
#                 not an order parameter" regime from earlier sessions.
# (hz, hx_c, Mx jump at hx_c, type)
MAG = {
    0.0: [(0.0, 0.9745, 0.0082, "topo"), (0.4, 0.8988, 0.2535, "trivtriv")],
    0.2: [(0.4, 0.8368, 0.2812, "trivtriv"), (0.7, 1.1158, 0.0907, "trivtriv")],
    0.4: [(0.2, 0.9067, 0.0043, "topo"), (0.4, 0.8267, 0.3397, "trivtriv"),
          (0.7, 1.1020, 0.0859, "trivtriv"), (1.0, 1.3475, 0.0168, "trivtriv")],
}

# hz=0.0 and hz=0.2 where NO energy crossing was found in the sampled hx range (up-branch
# didn't reach far enough) still had 7 clean up-branch O_FM points each -- fit them directly
# with the SAME logistic locator used on the electric line (tf.locate_all). This is NOT the
# same physical quantity as MAG's "topo" energy-crossing points: at hz=0.2/hy=0.4, where BOTH
# exist, the O_FM(up) sigmoid gives 0.8218 while the energy crossing gives 0.9067 -- a ~0.08
# gap. That matches the pattern already found at hz=0.0/hy=0.0 (O_FM(up) sigmoid ~0.82-0.84
# vs the true energy-crossing 0.9745): the sigmoid on the up branch alone tends to locate
# where the METASTABLE branch's own order parameter starts collapsing (a spinodal), which
# systematically undershoots the true equilibrium transition. Keep both, plotted differently
# -- these are real, usable data, just answering a different question than "topo" above.
# (hz, hx_c, hx_c_err)
MAG_SPINODAL = {
    0.0: [(0.2, 0.8352, 0.0051)],
    0.2: [],   # both points here superseded 2026-09-15 by HY02_TIP (user-dictated values)
    0.4: [(0.0, 0.8064, 0.0048), (0.2, 0.8218, 0.0044)],   # hz=0.2 here ALSO has the equilibrium point (0.9067) in MAG
}

# EXPLICIT USER OVERRIDE, hy=0.2, 2026-09-15: user-dictated (not recomputed) magnetic
# boundary points, connecting the electric segment to the trivial-trivial line's closest
# point (hz, hx_c). No error bars given -- plotted without them.
HY02_TIP = [(0.0, 0.815), (0.2, 0.819)]

# No crossing found and no O_FM(up) evidence either way (chain too short / not launched
# there at all -- genuinely nothing to fit, unlike the MAG_SPINODAL cases above).
MAG_UNRESOLVED = {0.0: [0.7, 1.0], 0.2: [1.0]}

for hy, pts in L4_HY.items():
    print(f"hy={hy}: " + ", ".join(f"hx={hx} hz_c={hz:.4f}+-{hze:.4f}" for hx, hz, hze in pts))
    for hz, hxc, jump, typ in MAG.get(hy, []):
        print(f"       magnetic hz={hz}: hx_c={hxc:.4f}  Mx_jump={jump:.4f}  [{typ}, equilibrium]")
    for hz, hxc, hxce in MAG_SPINODAL.get(hy, []):
        print(f"       magnetic hz={hz}: hx_c={hxc:.4f}+-{hxce:.4f}  [spinodal, O_FM(up) sigmoid]")
    if MAG_UNRESOLVED.get(hy):
        print(f"       magnetic hz={MAG_UNRESOLVED[hy]}: no crossing, no O_FM evidence -- unresolved")

In [ ]:
ORANGE = "#c9822a"
GREY_SPIN = "0.55"

fig, axes = plt.subplots(1, 3, figsize=(14, 5.2), sharex=True, sharey=True)
XLIM, YLIM = (0, 1.05), (-0.06, 1.5)

for ax, hy in zip(axes, (0.0, 0.2, 0.4)):
    ax.set_facecolor(GREY)

    # pocket boundary: electric (3 points) + magnetic "topo" EQUILIBRIUM points, sorted by hx
    # (hx increases monotonically along this whole path even though hz loops back near the
    # tip -- that loop-back IS the pocket narrowing to a point, not a plotting bug).
    # hy=0.2 gets the user-dictated HY02_TIP points appended instead of an equilibrium point.
    extra = [(hxc, hz, 0.0) for hz, hxc, _, typ in MAG.get(hy, []) if typ == "topo"]
    if hy == 0.2:
        extra += [(hxc, hz, 0.0) for hz, hxc in HY02_TIP]
    pts = list(L4_HY[hy]) + extra
    hx_c = np.array([p[0] for p in pts]); hz_c = np.array([p[1] for p in pts])
    hz_ce = np.array([p[2] for p in L4_HY[hy]] + [0.0] * (len(pts) - len(L4_HY[hy])))
    order = np.argsort(hx_c)
    hzo, hxo = hz_c[order], hx_c[order]
    ax.plot(hzo, hxo, "--", color="k", lw=1.6, zorder=3)
    # closed polygon (not fill_betweenx): the boundary loops back near the tip (hz drops
    # then rises again as hx keeps increasing), so it's not a simple function of hx -- a
    # per-row fill_betweenx pinches/self-overlaps there instead of showing the true notch.
    poly_hz = np.concatenate([[0.0], hzo, [0.0]])
    poly_hx = np.concatenate([[hxo[0]], hxo, [hxo[-1]]])
    ax.fill(poly_hz, poly_hx, color=PURPLE, zorder=1)
    ax.errorbar(hz_c, hx_c, xerr=hz_ce, fmt="o", ms=8, color="k", ecolor="k", elinewidth=1.3,
                capsize=3, zorder=4)

    # spinodal front: O_FM(up)-sigmoid points not yet promoted to the boundary -- own light
    # grey dotted line/marker (hy=0.2 has none left; both were promoted into HY02_TIP above)
    sp = sorted([(hz, hxc, hxce) for hz, hxc, hxce in MAG_SPINODAL.get(hy, [])])
    if sp:
        shz = np.array([p[0] for p in sp]); shx = np.array([p[1] for p in sp]); shxe = np.array([p[2] for p in sp])
        if len(sp) >= 2:
            ax.plot(shz, shx, ":", color=GREY_SPIN, lw=1.6, zorder=2)
        ax.errorbar(shz, shx, yerr=shxe, fmt="D", ms=6.5, mfc="0.8", mec=GREY_SPIN, mew=1.2,
                    ecolor=GREY_SPIN, capsize=2, zorder=3)

    # trivial-trivial boundary: hy=0.2 gets a leading segment from the closest tip point
    # (HY02_TIP's hz=0.2 point, per user instruction) into the first real trivtriv point
    tt = sorted([(hz, hxc) for hz, hxc, _, typ in MAG.get(hy, []) if typ == "trivtriv"])
    if hy == 0.2:
        tip_hx, tip_hz, _ = L4_HY[0.2][2]   # electric hx=0.8 -- more trustworthy than the
    else:                                    # magnetic spinodal points here, so IT is the tip
        tip_hz, tip_hx = hzo[-1], hxo[-1]    # magnetic "topo" equilibrium point (real L4
    tt = [(tip_hz, tip_hx)] + tt             # energy-crossing) -- already the outermost point
    thz, thx = np.array([p[0] for p in tt]), np.array([p[1] for p in tt])
    ax.plot(thz, thx, "-", color=ORANGE, lw=1.8, zorder=3)
    mhz, mhx = thz[1:], thx[1:]   # don't double-draw the shared tip point (already black)
    if len(mhz):
        ax.plot(mhz, mhx, "o", ms=8, color=ORANGE, mec="k", mew=0.6, zorder=4)

    ax.text(0.04, 0.15, "Top.", fontsize=12, fontweight="bold", color="white")
    ax.text(0.55, 0.55, "Trivial", fontsize=12, fontweight="bold", color="0.15")
    ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
    ax.set_xlabel("$h_z$")
    ax.set_title(f"$h_y$ = {hy:g}", fontsize=13, fontweight="bold")
axes[0].set_ylabel("$h_x$", fontsize=13)
handles = [plt.Line2D([], [], marker="o", ls="--", color="k", label="pocket boundary"),
           plt.Line2D([], [], marker="D", ls=":", color=GREY_SPIN, mfc="0.8", label="spinodal front (lower bound)"),
           plt.Line2D([], [], marker="o", ls="-", color=ORANGE, label="trivial-trivial boundary")]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, -0.05))
fig.suptitle("3D toric code -- L=4, all cuts, three $h_y$ planes", fontsize=14, fontweight="bold", y=1.03)
fig.tight_layout()
plt.show()

**Remaining true gaps (not plotted -- no y-value exists, unlike `MAG_SPINODAL` above which
IS now used):** `MAG_UNRESOLVED` -- no crossing, no O_FM(up) evidence either way, chains too
short/not launched there at all: hz=0.7/1.0 at hy=0.0, hz=1.0 at hy=0.2. Printed explicitly
by the data cell above every time it runs, not silently dropped.

---

# Part 3 — 3D topological pocket + trivial-trivial boundary (L=4)

Electric family: regular 3 (hx) x 3 (hy) grid -> a real surface (the purple lid) -- now
elongated in hx after promoting hx=0.8 (see the note in the data cell above: the
independent magnetic-line hz=0.2 crossing at hx_c=0.9067 backs up a genuine bulge, not a
fitting artifact). Magnetic family: irregular coverage (not every hz landed a crossing at
every hy) -> shown as points and, where >=2 hz values exist at one hy, connected lines, not
a forced surface.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 -- registers the 3d projection
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

HX_GRID = np.array([0.0, 0.5, 0.8])
HY_GRID = np.array([0.0, 0.2, 0.4])
Z = np.array([[L4_HY[hy][i][1] for hy in HY_GRID] for i in range(3)])   # Z[hx_idx, hy_idx]
HXm, HYm = np.meshgrid(HX_GRID, HY_GRID, indexing="ij")

# magnetic points, flattened across hy, split by type
mtx, mty, mtz = [], [], []   # topo (equilibrium)
mvx, mvy, mvz = [], [], []   # trivtriv
for hy, rows in MAG.items():
    for hz, hxc, jump, typ in rows:
        (mtx if typ == "topo" else mvx).append(hxc)
        (mty if typ == "topo" else mvy).append(hy)
        (mtz if typ == "topo" else mvz).append(hz)
spx, spy, spz = [], [], []   # spinodal (O_FM(up) sigmoid, lower bound)
for hy, rows in MAG_SPINODAL.items():
    for hz, hxc, hxce in rows:
        spx.append(hxc); spy.append(hy); spz.append(hz)

ZFLOOR = 0.0   # hz=0 is the physical floor (h=0 stabilizer point), not an auto-scaled min
ZTOP = max(Z.max(), max(mtz, default=0), max(mvz, default=0), max(spz, default=0)) + 0.05

fig = plt.figure(figsize=(9, 7.5))
ax = fig.add_subplot(111, projection="3d")

# floor shadow: footprint of the ELECTRIC (hx, hy) domain at hz=0
floor_x = [HX_GRID.min(), HX_GRID.max(), HX_GRID.max(), HX_GRID.min()]
floor_y = [HY_GRID.min(), HY_GRID.min(), HY_GRID.max(), HY_GRID.max()]
ax.add_collection3d(Poly3DCollection([list(zip(floor_x, floor_y, [ZFLOOR] * 4))],
                                     facecolor="0.85", edgecolor="0.6", alpha=0.5, zorder=1))
for i in range(HXm.shape[0]):
    for j in range(HXm.shape[1]):
        ax.plot([HXm[i, j]] * 2, [HYm[i, j]] * 2, [ZFLOOR, Z[i, j]], ":", color="0.55", lw=1, zorder=2)

ax.plot_surface(HXm, HYm, Z, color=PURPLE, alpha=0.65, edgecolor="k", linewidth=0.7,
                 antialiased=True, zorder=3)
ax.scatter(HXm, HYm, Z, color="k", s=45, depthshade=False, zorder=5, label="electric (confirmed)")

# magnetic "topo" (equilibrium) points: same pocket boundary, black diamonds
for x, y, z in zip(mtx, mty, mtz):
    ax.plot([x, x], [y, y], [ZFLOOR, z], ":", color="0.55", lw=1, zorder=2)
if mtx:
    ax.scatter(mtx, mty, mtz, marker="D", s=70, color="k", edgecolor="w", linewidth=0.6,
               depthshade=False, zorder=6, label="magnetic (topo, equilibrium)")

# spinodal points: O_FM(up) sigmoid, a lower-bound companion to the equilibrium points --
# small light-grey diamonds, no drop-lines (would clutter; already visually "low")
if spx:
    ax.scatter(spx, spy, spz, marker="D", s=35, color="0.75", edgecolor="0.4", linewidth=0.6,
               depthshade=False, zorder=5, label="magnetic (spinodal, lower bound)")

# magnetic "trivtriv" points: a SEPARATE boundary (not part of the topological volume)
ORANGE = "#c9822a"
for x, y, z in zip(mvx, mvy, mvz):
    ax.plot([x, x], [y, y], [ZFLOOR, z], ":", color="#e3c9a4", lw=0.8, zorder=2)
if mvx:
    ax.scatter(mvx, mvy, mvz, marker="^", s=70, color=ORANGE, edgecolor="k", linewidth=0.6,
               depthshade=False, zorder=6, label="magnetic (trivial-trivial)")
for hy in HY_GRID:
    row = sorted([(hz, hxc) for hz, hxc, _, typ in MAG.get(hy, []) if typ == "trivtriv"])
    if len(row) >= 2:
        rz, rx = zip(*row)
        ax.plot(rx, [hy] * len(rx), rz, "-", color=ORANGE, lw=2.0, zorder=4)

ax.set_xlabel("$h_x$", labelpad=10); ax.set_ylabel("$h_y$", labelpad=10)
ax.set_zlabel("$h_z^c$", labelpad=6)
ax.set_zlim(ZFLOOR, ZTOP)
ax.set_title("Topological pocket + trivial-trivial boundary (L=4)", fontsize=13, fontweight="bold", pad=0)
ax.view_init(elev=20, azim=-70)
ax.legend(frameon=False, fontsize=7.5, loc="upper left")
fig.tight_layout()
plt.show()